In [1]:
import numpy as np
import matplotlib.pyplot as plt
from MMTModules import *
%matplotlib inline

In [2]:
#detector positions
path_to_positions = 'data_4x4.txt'

#frequencies represented in position data in GHz
freqs = [90,150]
freq1 = freqs[0]
freq2 = freqs[1]

num_det = 64    #number of detectors

#Map Parameters:
rescale = 1

N = 2000    #number of pixels in horizontal and vertical edges of maps

phys_pixel_size = 5300. #micro m
f_ratio = 1.9 #CMB-S4 SPLAT
aperture = 5.*(10**3) #mm
arcsec_per_pixel = (phys_pixel_size/(f_ratio*aperture))*206.3
deg_per_pixel = arcsec_per_pixel/3600.
deg_per_microm = deg_per_pixel/phys_pixel_size
pixel_size = deg_per_pixel    #size of each pixel in deg

#Main beam fwhm in deg
beam_fwhm_90 = 2.5 * 1/60.   #90 GHz for CMB-S4 SPLAT
beam_fwhm_150 = 1.6 * 1/60.  #150 GHz for SMB-S4 SPLAT

#Correlation Parameters:
perc_corr = 0.001    #percent at which the signals are correlated
TtoP_suppress = False #suppress the T->P leakage that results from imbalanced detectors along certain polarization angles

#Power Spectrum Binning Parameters:
delta_ell = 50    #bin width
ell_max = 5000    #maximum ell

#choose normalization of power spectra
## 0 normalizes all PS to 1 for use with biasing spectra
## 'TT', 'EE', 'BB' normalizes to a main spectrum to study leakage
choose_normalization = 'TT'

#sky decomposition in I, Q, U space
sky_decomp = [1.0, 0.0, 0.0]

In [3]:
#generate focal plane distribution
det_dict = generate_focal_plane_distribution(path_to_positions, num_det, freq1, freq2, rescale)

#Making coupling dictionary:
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}

# Scheme 1

In [4]:
#read pol A, pol B alternating across row/col, T read left --> right/ B read top --> bottom
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}
            
        if Freq1==Freq2:
            for det in det_dict[Freq1].keys():
                
                if Freq1==90:
                    if det==0:
                        coupling_dict[(Freq1,Freq2)][det] = 31
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = det-1
                
                if Freq1==150:
                    if det==0:
                        coupling_dict[(Freq1,Freq2)][det] = 31
                    
                    elif det_dict[Freq1][det]['ID'][1]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det+23
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    
                    else:
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    
                    
                        
                        
        if Freq1<Freq2:
            for det in det_dict[Freq1].keys():
                coupling_dict[(Freq1,Freq2)][det] = 'na'
                    
        if Freq1>Freq2:
            for det in det_dict[Freq1].keys():
                coupling_dict[(Freq1,Freq2)][det] = 'na'


#calculate crosstalk 3x3 IQU beams
convolved_beam_matrix_1 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq1, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization, make_plot=False)
convolved_beam_matrix_2 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq2, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization, make_plot=False)
convolved_beam_matrix_3 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq2, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization, make_plot=False)
convolved_beam_matrix_4 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq1, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization, make_plot=False)

binned_ell_lr_tb1, binned_spectra_lr_tb1, binned_beam_lr_tb1 = get_leakage_spectra(convolved_beam_matrix_1, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_lr_tb2, binned_spectra_lr_tb2, binned_beam_lr_tb2 = get_leakage_spectra(convolved_beam_matrix_2, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_lr_tb3, binned_spectra_lr_tb3, binned_beam_lr_tb3 = get_leakage_spectra(convolved_beam_matrix_3, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_lr_tb4, binned_spectra_lr_tb4, binned_beam_lr_tb4 = get_leakage_spectra(convolved_beam_matrix_4, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)


# Scheme 2

In [5]:
#read pol A, pol B alternating across row/col, row/col 1:left --> right/top --> bottom, row/col 2:right --> left/bottom --> top
# 'snaking' readout order
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}
            
        if Freq1==Freq2:
            for det in det_dict[Freq1].keys():
                
                if Freq1==90:
                    if det==0:
                        coupling_dict[(Freq1,Freq2)][det] = 25
                    
                    elif det_dict[Freq1][det]['ID'][1]=='0':
                        coupling_dict[(Freq1,Freq2)][det] = det-1
                        
                    elif det_dict[Freq1][det]['ID'][1]=='2':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-7
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1

                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    
                    else:
                        if det_dict[Freq1][det]['ID'][2]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-7
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                    
                        else:
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det+3
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                        
                
                if Freq1==150:
                    if det==0:
                        coupling_dict[(Freq1,Freq2)][det] = 7
                    
                    elif det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                            
                    elif det_dict[Freq1][det]['ID'][2]=='2':
                        if det_dict[Freq1][det]['ID'][1]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1


                        else:
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-7
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                                
                    else:
                        if det_dict[Freq1][det]['ID'][1]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1

                        else:
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det+9
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                       
                        
        if Freq1<Freq2:
            for det in det_dict[Freq1].keys():
                coupling_dict[(Freq1,Freq2)][det] = 'na'
                    
        if Freq1>Freq2:
            for det in det_dict[Freq1].keys():
                coupling_dict[(Freq1,Freq2)][det] = 'na'

#calculate crosstalk 3x3 IQU beams
convolved_beam_matrix_1 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq1, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_2 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq2, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_3 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq2, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_4 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq1, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)

binned_ell_snake1, binned_spectra_snake1, binned_beam_snake1 = get_leakage_spectra(convolved_beam_matrix_1, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_snake2, binned_spectra_snake2, binned_beam_snake2 = get_leakage_spectra(convolved_beam_matrix_2, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_snake3, binned_spectra_snake3, binned_beam_snake3 = get_leakage_spectra(convolved_beam_matrix_3, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_snake4, binned_spectra_snake4, binned_beam_snake4 = get_leakage_spectra(convolved_beam_matrix_4, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)

# Scheme 3

In [6]:
#read all pol A across row/col, left --> right/top --> bottom, , then all pol B across row/col, left --> right/top --> bottom
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}
            
        if Freq1==Freq2:
            for det in det_dict[Freq1].keys():
                
                if Freq1==90:
                    if det==0:
                        coupling_dict[(Freq1,Freq2)][det] = 31
                        
                    elif det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det+5
                   
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = det-2
                        
                
                if Freq1==150:
                    if det==0:
                        coupling_dict[(Freq1,Freq2)][det] = 31
                        
                    elif det_dict[Freq1][det]['ID'][1]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det+23
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det+23
                        
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = det-8
                   
                        
        if Freq1<Freq2:
            for det in det_dict[Freq1].keys():
                coupling_dict[(Freq1,Freq2)][det] = 'na'
                    
        if Freq1>Freq2:
            for det in det_dict[Freq1].keys():
                coupling_dict[(Freq1,Freq2)][det] = 'na'

#calculate crosstalk 3x3 IQU beams
convolved_beam_matrix_1 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq1, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_2 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq2, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_3 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq2, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_4 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq1, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)

binned_ell_one_pol_lr_tb1, binned_spectra_one_pol_lr_tb1, binned_beam_one_pol_lr_tb1 = get_leakage_spectra(convolved_beam_matrix_1, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_one_pol_lr_tb2, binned_spectra_one_pol_lr_tb2, binned_beam_one_pol_lr_tb2 = get_leakage_spectra(convolved_beam_matrix_2, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_one_pol_lr_tb3, binned_spectra_one_pol_lr_tb3, binned_beam_one_pol_lr_tb3 = get_leakage_spectra(convolved_beam_matrix_3, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_one_pol_lr_tb4, binned_spectra_one_pol_lr_tb4, binned_beam_one_pol_lr_tb4 = get_leakage_spectra(convolved_beam_matrix_4, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)

# Scheme 4

In [7]:
#read all pol A across row/col, left --> right/top --> bottom, , then all pol B across row/col, right --> left/bottom --> top
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}
            
        if Freq1==Freq2:
            for det in det_dict[Freq1].keys():
                
                if Freq1==90:
                    if det==0:
                        coupling_dict[(Freq1,Freq2)][det] = 25
                    
                    elif det_dict[Freq1][det]['ID'][1]=='0':
                        if  det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2!=0:
                                coupling_dict[(Freq1,Freq2)][det] = det+5
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-2
                            
                        
                    elif det_dict[Freq1][det]['ID'][1]=='1':
                        if  det_dict[Freq1][det]['ID'][2]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-7
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-7
                                
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det+2
                            
                    elif det_dict[Freq1][det]['ID'][1]=='3':
                        if  det_dict[Freq1][det]['ID'][2]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-7
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-7
                                
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det+2
                            
                    else:
                        if  det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-7
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det+5
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-2
                        
                
                if Freq1==150:
                    if det==0:
                        coupling_dict[(Freq1,Freq2)][det] = 7
                    
                    elif det_dict[Freq1][det]['ID'][2]=='0':
                        if det_dict[Freq1][det]['ID'][1]=='0':
                            if det%2!=0:
                                coupling_dict[(Freq1,Freq2)][det] = det+23
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-8
                            
                    elif det_dict[Freq1][det]['ID'][2]=='2':
                        if det_dict[Freq1][det]['ID'][1]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det+23


                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-8
                            
                                
                    else:
                        if det_dict[Freq1][det]['ID'][1]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-25

                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det+8
                       
                        
        if Freq1<Freq2:
            for det in det_dict[Freq1].keys():
                coupling_dict[(Freq1,Freq2)][det] = 'na'
                    
        if Freq1>Freq2:
            for det in det_dict[Freq1].keys():
                coupling_dict[(Freq1,Freq2)][det] = 'na'
#calculate crosstalk 3x3 IQU beams
convolved_beam_matrix_1 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq1, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_2 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq2, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_3 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq2, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_4 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq1, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)

binned_ell_one_pol_snake1, binned_spectra_one_pol_snake1, binned_beam_one_pol_snake1 = get_leakage_spectra(convolved_beam_matrix_1, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_one_pol_snake2, binned_spectra_one_pol_snake2, binned_beam_one_pol_snake2 = get_leakage_spectra(convolved_beam_matrix_2, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_one_pol_snake3, binned_spectra_one_pol_snake3, binned_beam_one_pol_snake3 = get_leakage_spectra(convolved_beam_matrix_3, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_one_pol_snake4, binned_spectra_one_pol_snake4, binned_beam_one_pol_snake4 = get_leakage_spectra(convolved_beam_matrix_4, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)

# Scheme 5

In [8]:
# all frequencies read out in same readout column - all 95, then all 150
#read pol A, pol B alternating across row left --> right
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}
            
        if Freq1==Freq2:
            for det in det_dict[Freq1].keys():
                
                if Freq1==90:
                        
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = det-1
                        
                
                if Freq1==150:
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = det-1
                       
                        
        if Freq1<Freq2:
            for det in det_dict[Freq1].keys():
                if det==0:
                    coupling_dict[(Freq1,Freq2)][det] = 31
                    
                elif det_dict[Freq1][det]['ID'][2]=='0':
                    if det%2==0:
                        coupling_dict[(Freq1,Freq2)][det] = det-1
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                else:
                    coupling_dict[(Freq1,Freq2)][det] = 'na'
                    
        if Freq1>Freq2:
            for det in det_dict[Freq1].keys():
                if det_dict[Freq1][det]['ID'][2]=='0':
                    if det%2==0:
                        coupling_dict[(Freq1,Freq2)][det] = det+7
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                else:
                    coupling_dict[(Freq1,Freq2)][det] = 'na'

#calculate crosstalk 3x3 IQU beams
convolved_beam_matrix_1 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq1, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_2 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq2, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_3 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq2, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_4 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq1, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)

binned_ell_all_freq_lr1, binned_spectra_all_freq_lr1, binned_beam_all_freq_lr1 = get_leakage_spectra(convolved_beam_matrix_1, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_lr2, binned_spectra_all_freq_lr2, binned_beam_all_freq_lr2 = get_leakage_spectra(convolved_beam_matrix_2, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_lr3, binned_spectra_all_freq_lr3, binned_beam_all_freq_lr3 = get_leakage_spectra(convolved_beam_matrix_3, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_lr4, binned_spectra_all_freq_lr4, binned_beam_all_freq_lr4 = get_leakage_spectra(convolved_beam_matrix_4, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)

# Scheme 6 

In [9]:
# all frequencies read out in same readout column - all 95, then all 150
#read pol A, pol B alternating across row by 'snaking'
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}
            
        if Freq1==Freq2:
            for det in det_dict[Freq1].keys():
                
                if Freq1==90:
                        
                    if det_dict[Freq1][det]['ID'][1]=='0':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                 coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                            
                    elif det_dict[Freq1][det]['ID'][1]=='2':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                 coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    
                    else:
                        if det_dict[Freq1][det]['ID'][2]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                 coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det+3
                            else:
                                 coupling_dict[(Freq1,Freq2)][det] = det-1
                
                if Freq1==150:
                    if det_dict[Freq1][det]['ID'][1]=='0':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                 coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                            
                    elif det_dict[Freq1][det]['ID'][1]=='2':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                 coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    
                    else:
                        if det_dict[Freq1][det]['ID'][2]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                 coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = det+3
                            else:
                                 coupling_dict[(Freq1,Freq2)][det] = det-1
                        
                        
            
                       
                        
        if Freq1<Freq2:
            for det in det_dict[Freq1].keys():
                if det==0:
                    coupling_dict[(Freq1,Freq2)][det] = 25
                    
                elif det_dict[Freq1][det]['ID'][1]=='0':
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
                elif det_dict[Freq1][det]['ID'][1]=='2':
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
                else:
                    if det_dict[Freq1][det]['ID'][2]=='3':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
            
                        
        if Freq1>Freq2:
            for det in det_dict[Freq1].keys():
                if det_dict[Freq1][det]['ID'][1]=='0':
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det+7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
                elif det_dict[Freq1][det]['ID'][1]=='2':
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det+7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
                else:
                    if det_dict[Freq1][det]['ID'][2]=='3':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-5
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'

#calculate crosstalk 3x3 IQU beams
convolved_beam_matrix_1 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq1, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_2 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq2, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_3 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq2, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_4 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq1, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)

binned_ell_all_freq_lr_snake1, binned_spectra_all_freq_lr_snake1, binned_beam_all_freq_lr_snake1 = get_leakage_spectra(convolved_beam_matrix_1, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_lr_snake2, binned_spectra_all_freq_lr_snake2, binned_beam_all_freq_lr_snake2 = get_leakage_spectra(convolved_beam_matrix_2, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_lr_snake3, binned_spectra_all_freq_lr_snake3, binned_beam_all_freq_lr_snake3 = get_leakage_spectra(convolved_beam_matrix_3, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_lr_snake4, binned_spectra_all_freq_lr_snake4, binned_beam_all_freq_lr_snake4 = get_leakage_spectra(convolved_beam_matrix_4, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)

# Scheme 7

In [10]:
# all frequencies read out in same readout column - alternating between 90 polA+B and 150 polA+B
#read pol A, pol B alternating across row left --> right
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}
            
        if Freq1==Freq2:
            for det in det_dict[Freq1].keys():
                if Freq1==90:
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    else:
                        if det%2!=0:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                
                
                if Freq1==150:
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                    else:
                        if det%2!=0:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                       
                        
        if Freq1<Freq2:
            for det in det_dict[Freq1].keys():
                if det==0:
                    coupling_dict[(Freq1,Freq2)][det] = 31
                
                elif det%2==0:
                    coupling_dict[(Freq1,Freq2)][det] = det-1
                else:
                    coupling_dict[(Freq1,Freq2)][det] = 'na'
                
            
                        
        if Freq1>Freq2:
            for det in det_dict[Freq1].keys():
                if det%2==0:
                    coupling_dict[(Freq1,Freq2)][det] = det+1
                else:
                    coupling_dict[(Freq1,Freq2)][det] = 'na'

#calculate crosstalk 3x3 IQU beams
convolved_beam_matrix_1 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq1, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_2 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq2, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_3 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq2, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_4 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq1, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)

binned_ell_all_freq_alt1, binned_spectra_all_freq_alt1, binned_beam_all_freq_alt1 = get_leakage_spectra(convolved_beam_matrix_1, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_alt2, binned_spectra_all_freq_alt2, binned_beam_all_freq_alt2 = get_leakage_spectra(convolved_beam_matrix_2, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_alt3, binned_spectra_all_freq_alt3, binned_beam_all_freq_alt3 = get_leakage_spectra(convolved_beam_matrix_3, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_alt4, binned_spectra_all_freq_alt4, binned_beam_all_freq_alt4 = get_leakage_spectra(convolved_beam_matrix_4, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)

# Scheme 8

In [11]:
# all frequencies read out in same readout column - alternating between 90 polA+B and 150 polA+B
#read pol A, pol B alternating across row 1: left --> right, row2: right --> left
# 'snaking' readout
freqs = det_dict.keys()
coupling_dict = {}
for Freq1 in freqs:
    for Freq2 in freqs:
        coupling_dict[(Freq1,Freq2)] = {}
            
        if Freq1==Freq2:
            for det in det_dict[Freq1].keys():
                if Freq1==90:
                    if det_dict[Freq1][det]['ID'][1]=='0':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            if det%2!=0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                                
                    elif det_dict[Freq1][det]['ID'][1]=='2':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            if det%2!=0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                                
                    else:
                        if det_dict[Freq1][det]['ID'][2]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            if det%2!=0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                
                
                if Freq1==150:
                    if det_dict[Freq1][det]['ID'][1]=='0':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            if det%2!=0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            
                    elif det_dict[Freq1][det]['ID'][1]=='2':
                        if det_dict[Freq1][det]['ID'][2]=='0':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            if det%2!=0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            
                    else:
                        if det_dict[Freq1][det]['ID'][2]=='3':
                            if det%2==0:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            if det%2!=0:
                                coupling_dict[(Freq1,Freq2)][det] = det-1
                            else:
                                coupling_dict[(Freq1,Freq2)][det] = 'na'
                       
                        
        if Freq1<Freq2:
            for det in det_dict[Freq1].keys():
                if det==0:
                    coupling_dict[(Freq1,Freq2)][det] = 25
                    
                elif det_dict[Freq1][det]['ID'][1]=='0':
                    if det%2==0:
                        coupling_dict[(Freq1,Freq2)][det] = det-1
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
                elif det_dict[Freq1][det]['ID'][1]=='2':
                    if det_dict[Freq1][det]['ID'][2]=='0':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                    else:        
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-1
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
                else:
                    if det_dict[Freq1][det]['ID'][2]=='3':
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det-7
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
                    else:
                        if det%2==0:
                            coupling_dict[(Freq1,Freq2)][det] = det+3
                        else:
                            coupling_dict[(Freq1,Freq2)][det] = 'na'
            
                        
        if Freq1>Freq2:
            for det in det_dict[Freq1].keys():
                if det_dict[Freq1][det]['ID'][1]=='0':
                    if det%2==0:
                        coupling_dict[(Freq1,Freq2)][det] = det+1
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
                elif det_dict[Freq1][det]['ID'][1]=='2':
                    if det%2==0:
                        coupling_dict[(Freq1,Freq2)][det] = det+1
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'
                        
                else:
                    if det%2==0:
                        coupling_dict[(Freq1,Freq2)][det] = det+1
                    else:
                        coupling_dict[(Freq1,Freq2)][det] = 'na'

#calculate crosstalk 3x3 IQU beams
convolved_beam_matrix_1 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq1, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_2 = calculate_crosstalk(det_dict, coupling_dict, freq1,freq2, pixel_size, perc_corr, N, beam_fwhm_90, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_3 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq2, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)
convolved_beam_matrix_4 = calculate_crosstalk(det_dict, coupling_dict, freq2,freq1, pixel_size, perc_corr, N, beam_fwhm_150, sky_decomp, TtoP_suppress, delta_ell, ell_max, choose_normalization)

binned_ell_all_freq_alt_snake1, binned_spectra_all_freq_alt_snake1, binned_beam_all_freq_alt_snake1 = get_leakage_spectra(convolved_beam_matrix_1, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_alt_snake2, binned_spectra_all_freq_alt_snake2, binned_beam_all_freq_alt_snake2 = get_leakage_spectra(convolved_beam_matrix_2, pixel_size, N, beam_fwhm_90, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_alt_snake3, binned_spectra_all_freq_alt_snake3, binned_beam_all_freq_alt_snake3 = get_leakage_spectra(convolved_beam_matrix_3, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)
binned_ell_all_freq_alt_snake4, binned_spectra_all_freq_alt_snake4, binned_beam_all_freq_alt_snake4 = get_leakage_spectra(convolved_beam_matrix_4, pixel_size, N, beam_fwhm_150, sky_decomp, delta_ell, ell_max, choose_normalization)

In [12]:
np.save('sim_data/lr_tb1_setTime_BB.npy',binned_spectra_lr_tb1['BB'])
np.save('sim_data/snake1_setTime_BB.npy',binned_spectra_snake1['BB'])
np.save('sim_data/one_pol_lr_tb1_setTime_BB.npy',binned_spectra_one_pol_lr_tb1['BB'])
np.save('sim_data/one_pol_snake1_setTime_BB.npy',binned_spectra_one_pol_snake1['BB'])

np.save('sim_data/lr_tb2_setTime_BB.npy',binned_spectra_lr_tb2['BB'])
np.save('sim_data/snake2_setTime_BB.npy',binned_spectra_snake2['BB'])
np.save('sim_data/one_pol_lr_tb2_setTime_BB.npy',binned_spectra_one_pol_lr_tb2['BB'])
np.save('sim_data/one_pol_snake2_setTime_BB.npy',binned_spectra_one_pol_snake2['BB'])

np.save('sim_data/lr_tb3_setTime_BB.npy',binned_spectra_lr_tb3['BB'])
np.save('sim_data/snake3_setTime_BB.npy',binned_spectra_snake3['BB'])
np.save('sim_data/one_pol_lr_tb3_setTime_BB.npy',binned_spectra_one_pol_lr_tb3['BB'])
np.save('sim_data/one_pol_snake3_setTime_BB.npy',binned_spectra_one_pol_snake3['BB'])

np.save('sim_data/lr_tb4_setTime_BB.npy',binned_spectra_lr_tb4['BB'])
np.save('sim_data/snake4_setTime_BB.npy',binned_spectra_snake4['BB'])
np.save('sim_data/one_pol_lr_tb4_setTime_BB.npy',binned_spectra_one_pol_lr_tb4['BB'])
np.save('sim_data/one_pol_snake4_setTime_BB.npy',binned_spectra_one_pol_snake4['BB'])

np.save('sim_data/both_lr_tb1_setTime_BB.npy',binned_spectra_lr_tb1['BB'])
np.save('sim_data/both_snake1_setTime_BB.npy',binned_spectra_snake1['BB'])
np.save('sim_data/both_all_freq_alt1_setTime_BB.npy',binned_spectra_all_freq_alt1['BB'])
np.save('sim_data/both_all_freq_alt_snake1_setTime_BB.npy',binned_spectra_all_freq_alt_snake1['BB'])

np.save('sim_data/both_lr_tb2_setTime_BB.npy',binned_spectra_lr_tb2['BB'])
np.save('sim_data/both_snake2_setTime_BB.npy',binned_spectra_snake2['BB'])
np.save('sim_data/both_all_freq_alt2_setTime_BB.npy',binned_spectra_all_freq_alt2['BB'])
np.save('sim_data/both_all_freq_alt_snake2_setTime_BB.npy',binned_spectra_all_freq_alt_snake2['BB'])

np.save('sim_data/both_lr_tb3_setTime_BB.npy',binned_spectra_lr_tb3['BB'])
np.save('sim_data/both_snake3_setTime_BB.npy',binned_spectra_snake3['BB'])
np.save('sim_data/both_all_freq_alt3_setTime_BB.npy',binned_spectra_all_freq_alt3['BB'])
np.save('sim_data/both_all_freq_alt_snake3_setTime_BB.npy',binned_spectra_all_freq_alt_snake3['BB'])

np.save('sim_data/both_lr_tb4_setTime_BB.npy',binned_spectra_lr_tb4['BB'])
np.save('sim_data/both_snake4_setTime_BB.npy',binned_spectra_snake4['BB'])
np.save('sim_data/both_all_freq_alt4_setTime_BB.npy',binned_spectra_all_freq_alt4['BB'])
np.save('sim_data/both_all_freq_alt_snake4_setTime_BB.npy',binned_spectra_all_freq_alt_snake4['BB'])